# Planner and Executor:

There are 2 agents:

- Planner Agent — the planner_node function, which uses the LLM to decompose the task into ordered steps.
- Executor Agent — the executor_node function, which loops through each step and routes it to the correct tool.

## Architecture
```
The agent is a LangGraph state machine with three nodes:
Task ──▶ [Planner] ──▶ [Executor] ──▶ ... loop ... ──▶ [Finalizer] ──▶ Output


1. Planner node — 
Makes one LLM call. Asks Claude to decompose the task into exactly 4 ordered steps, returned as a JSON array. Each step starts with a keyword (search, extract, outline, write) so the executor can route it.

2. Executor node (loops) — 
Reads the current step index, looks up the matching tool by keyword, calls it with the right inputs (including passing previous step outputs forward), appends the result to step_results, and increments the index. A conditional edge checks: are there more steps? If yes → loop back; if no → move on.

3. Finalizer node — Packages the last step result as final_output.

In [2]:
"""
Planner + Executor Agent using LangGraph
=========================================
The Planner breaks a high-level task into ordered steps.
The Executor runs each step one at a time, calling real tools.
"""

import os
import json
from typing import TypedDict, Annotated, List, Optional
import operator

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

In [7]:
# 2. Initialize the LLM


# OPEN_API_KEY = os.environ["OPEN_API_KEY"]
OPEN_API_KEY= ""

# llm = ChatAnthropic(
#     model="claude-sonnet-4-6",
#     api_key=ANTHROPIC_API_KEY,
#     max_tokens=4096,
# )

llm = ChatOpenAI(
    model="gpt-4o", 
    api_key=OPEN_API_KEY,
    max_tokens=4096,
)


# Test the LLM with a simple prompt
try:
    print("Testing connection to LLM...")
    response = llm.invoke("Hello! Confirm you are working by replying with 'System online'.")
    print("\n--- Response ---")
    print(response.content)
    print("----------------")
except Exception as e:
    print(f"\nAn error occurred: {e}")

Testing connection to LLM...

--- Response ---
System online.
----------------


In [10]:
# 2.  Shared graph state
class AgentState(TypedDict):
    task: str                            # The original user task
    plan: List[str]                      # Ordered list of step descriptions
    current_step_index: int              # Which step we're executing
    step_results: Annotated[List[str], operator.add]  # Accumulated outputs
    final_output: Optional[str]          # The finished blog post / answer
    error: Optional[str]                 # Any error message


# ─────────────────────────────────────────────
# 3.  Tool implementations
# ─────────────────────────────────────────────
@tool
def search_ai_trends(query: str) -> str:
    """Search for the latest AI trends and news."""
    # In production wire up Tavily / SerpAPI here.
    # For demo purposes we return rich, realistic mock data.
    return """
        Latest AI Trends (2025):
        1. Agentic AI Systems – Multi-step autonomous agents that plan and execute complex tasks without constant human intervention.
        2. Multimodal AI – Models handling text, images, audio, and video in a single unified architecture (GPT-4o, Gemini 2.0).
        3. On-Device / Edge AI – Smaller, efficient models (Phi-3, Gemma) running directly on phones and laptops.
        4. AI Reasoning Models – Chain-of-thought and 'test-time compute' approaches (OpenAI o3, DeepSeek-R1) achieving near-human reasoning on math and coding.
        5. AI Code Assistants – Copilot, Cursor, and Claude Code changing how software is written and reviewed.
        6. Open-Source Surge – Meta Llama 3, Mistral, and Qwen models closing the gap with proprietary systems.
        7. RAG Evolution – Retrieval-Augmented Generation moving from basic vector search to graph-RAG and agentic retrieval.
        8. AI Safety & Alignment – Anthropic's Constitutional AI, OpenAI's RLHF advances, and government regulation frameworks (EU AI Act now live).
        9. Synthetic Data – Using AI-generated data to train and fine-tune models, reducing dependence on labelled datasets.
        10. Enterprise AI Adoption – Companies integrating AI into every workflow; ROI measurement and governance frameworks become standard.
    """


@tool
def extract_key_points(text: str) -> str:
    """Extract and structure the key points from raw research text."""
    prompt = f"""
        You are a research analyst. Extract the 5 most important and compelling key points from the text below.
        For each point, provide:
        - A short headline
        - One supporting sentence
        
        Return plain text, no markdown.
        
        TEXT:
        {text}
    """
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


@tool
def create_blog_outline(key_points: str, task: str) -> str:
    """Create a structured blog post outline from key points."""
    prompt = f"""
        You are a content strategist. Create a detailed blog post outline for the topic: "{task}".
        Use these key points as the basis:
        
        {key_points}
        
        Include:
        - A catchy title
        - Introduction (hook + thesis)
        - 3–5 main sections with sub-bullets
        - Conclusion with a call to action
        
        Return plain text.
    """
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


@tool
def write_blog_post(outline: str, key_points: str, task: str) -> str:
    """Write a full blog post from the outline and key points."""
    
    prompt = f"""
        You are a professional tech blogger. Write a complete, engaging blog post based on:
        
        TOPIC: {task}
        
        OUTLINE:
        {outline}
        
        KEY POINTS:
        {key_points}
        
        Guidelines:
        - 600–900 words
        - Conversational yet authoritative tone
        - Use clear section headings
        - End with an inspiring conclusion
        
        Write the full blog post now.
    """
    
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


# Map step keywords → tools
TOOL_REGISTRY = {
    "search":  search_ai_trends,
    "extract": extract_key_points,
    "outline": create_blog_outline,
    "write":   write_blog_post,
}


In [9]:
# 4.  Graph nodes

def planner_node(state: AgentState) -> AgentState:
    """Ask the LLM to decompose the task into concrete steps."""
    print("\n" + "═"*60)
    print("🧠  PLANNER — decomposing task …")
    print("═"*60)

    system = """You are a planning agent. Break the user's task into 4 concrete, ordered steps.
        Each step must begin with one of these action keywords: search, extract, outline, write.
        Return ONLY a JSON array of strings, e.g.:
        ["search for X", "extract key points from results", "outline the blog post", "write the full blog post"]
    """

    response = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"Task: {state['task']}"),
    ])

    raw = response.content.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    plan = json.loads(raw.strip())

    print("\n📋  Plan:")
    for i, step in enumerate(plan, 1):
        print(f"   Step {i}: {step}")

    return {**state, "plan": plan, "current_step_index": 0}


def executor_node(state: AgentState) -> AgentState:
    """Execute the current step by routing to the right tool."""
    idx  = state["current_step_index"]
    step = state["plan"][idx]

    print(f"\n{'─'*60}")
    print(f"⚙️   EXECUTOR — Step {idx + 1}/{len(state['plan'])}: {step}")
    print("─"*60)

    # Pick a tool by finding the first matching keyword in the step description
    chosen_tool = None
    for keyword, fn in TOOL_REGISTRY.items():
        if keyword in step.lower():
            chosen_tool = fn
            break

    if chosen_tool is None:
        result = f"[No tool matched for step: '{step}']"
    else:
        print(f"   🔧 Using tool: {chosen_tool.name}")

        # Build the tool's argument from previous results
        if chosen_tool.name == "search_ai_trends":
            result = chosen_tool.invoke({"query": state["task"]})

        elif chosen_tool.name == "extract_key_points":
            search_result = state["step_results"][-1] if state["step_results"] else ""
            result = chosen_tool.invoke({"text": search_result})

        elif chosen_tool.name == "create_blog_outline":
            key_points = state["step_results"][-1] if state["step_results"] else ""
            result = chosen_tool.invoke({"key_points": key_points, "task": state["task"]})

        elif chosen_tool.name == "write_blog_post":
            # grab key points (step 2 result) and outline (step 3 result)
            results = state["step_results"]
            key_points = results[-2] if len(results) >= 2 else ""
            outline    = results[-1] if results else ""
            result = chosen_tool.invoke({
                "outline":    outline,
                "key_points": key_points,
                "task":       state["task"],
            })
        else:
            result = "[Unknown tool]"

    # Preview the result
    preview = result[:300].replace("\n", " ") + ("…" if len(result) > 300 else "")
    print(f"\n   ✅ Result preview: {preview}")

    return {
        **state,
        "step_results":       [result],          # operator.add appends this
        "current_step_index": idx + 1,
    }


def finalizer_node(state: AgentState) -> AgentState:
    """Package everything into the final output."""
    print(f"\n{'═'*60}")
    print("🏁  FINALIZER — assembling output …")
    print("═"*60)

    # The last step result is the blog post
    final = state["step_results"][-1] if state["step_results"] else "No output generated."
    return {**state, "final_output": final}


# ─────────────────────────────────────────────
# 5.  Routing logic
# ─────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    """Route back to executor if there are remaining steps, else finalize."""
    if state["current_step_index"] < len(state["plan"]):
        return "execute"
    return "finalize"


# ─────────────────────────────────────────────
# 6.  Build the LangGraph graph
# ─────────────────────────────────────────────
def build_graph() -> StateGraph:
    graph = StateGraph(AgentState)

    graph.add_node("planner",   planner_node)
    graph.add_node("executor",  executor_node)
    graph.add_node("finalizer", finalizer_node)

    graph.set_entry_point("planner")

    graph.add_edge("planner", "executor")

    graph.add_conditional_edges(
        "executor",
        should_continue,
        {"execute": "executor", "finalize": "finalizer"},
    )

    graph.add_edge("finalizer", END)

    return graph.compile()


# ─────────────────────────────────────────────
# 7.  Entry point
# ─────────────────────────────────────────────
def run_agent(task: str) -> str:
    print("\n" + "█"*60)
    print(f"🚀  Task: {task}")
    print("█"*60)

    app = build_graph()

    initial_state: AgentState = {
        "task":              task,
        "plan":              [],
        "current_step_index": 0,
        "step_results":      [],
        "final_output":      None,
        "error":             None,
    }

    final_state = app.invoke(initial_state)

    print("\n" + "═"*60)
    print("📝  FINAL OUTPUT")
    print("═"*60)
    print(final_state["final_output"])

    return final_state["final_output"]


if __name__ == "__main__":
    run_agent("Summarize latest AI trends and create a blog post")


████████████████████████████████████████████████████████████
🚀  Task: Summarize latest AI trends and create a blog post
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
🧠  PLANNER — decomposing task …
════════════════════════════════════════════════════════════

📋  Plan:
   Step 1: search for the latest AI trends
   Step 2: extract key points from recent articles and reports
   Step 3: outline the structure of the blog post
   Step 4: write the full blog post based on the outline

────────────────────────────────────────────────────────────
⚙️   EXECUTOR — Step 1/4: search for the latest AI trends
────────────────────────────────────────────────────────────
   🔧 Using tool: search_ai_trends

   ✅ Result preview:  Latest AI Trends (2025): 1. Agentic AI Systems – Multi-step autonomous agents that plan and execute complex tasks without constant human intervention. 2. Multimodal AI – Models handling text, images, au

**Title:** Unpacking the Latest AI Trends: Autonomous Agents, Multimodal Marvels, & More

---

**Introduction**

- **Hook:** From self-driving cars to chatbots that seem to think like us, AI is no longer a futuristic promise—it's happening right now.
- **Thesis:** Today, we'll explore five groundbreaking AI trends that are reshaping how tasks are executed, data is integrated, reasoning is perfected, open source thrives, and enterprises transform.

---

**Main Sections**

1. **Agentic AI Systems Revolutionize Task Execution**

   - **Explanation of Agentic AI Systems:** Define what Agentic AI entails and how these systems function autonomously.
   - **Impact on Task Execution:** Discuss how these systems are optimized for multitasking across complex scenarios, reducing the need for constant human input.
   - **Implications for the Future:** Speculate on future possibilities and industries that will benefit most from agentic AI systems.

2. **Multimodal AI Integrates Diverse Data Modalities**

   - **Introduction to Multimodal AI Models:** Provide an overview of innovations like GPT-4o and Gemini 2.0.
   - **Capabilities Across Modalities:** Highlight the ability of these systems to interpret and interlink text, images, audio, and video.
   - **Real-world Applications:** Explore how multimodal AI is being applied in industries like healthcare, education, and entertainment.

3. **AI Reasoning Models Achieve Near-Human Capabilities**

   - **What Are AI Reasoning Models?:** Explain advanced reasoning models like OpenAI o3 and DeepSeek-R1.
   - **Capabilities in Mathematical and Coding Tasks:** Demonstrate how these models mimic human reasoning in specific domains.
   - **Potential for Human-Machine Collaboration:** Discuss the potential upsides of partnering human intellect with AI reasoning capabilities.

4. **Open-Source Models Close Gap with Proprietary Systems**

   - **Overview of the Open-Source Movement:** Talk about models like Meta Llama 3, Mistral, and Qwen.
   - **Performance and Accessibility:** Analyze how open-source AI models are rapidly improving and making sophisticated AI accessible to a broader audience.
   - **Community and Innovation:** Reflect on how the open-source community fosters innovation through collaboration.

5. **Enterprise AI Integration Becomes Standard**

   - **Current State of AI in Enterprises:** Explain the growing role of AI across business operations and decision-making.
   - **Bridging ROI with AI Governance:** Address how companies are implementing ROI measurements and governance frameworks.
   - **Preparing for Full Integration:** Offer advice on how businesses can further prepare for comprehensive AI integration.

---

**Conclusion**

- **Summarize Key Points:** Briefly recap the five key AI trends discussed.
- **Call to Action:** Encourage readers to stay informed and adaptive, whether they are AI enthusiasts, industry professionals, or business leaders seeking to leverage the benefits of AI in their own domains.

---

This outline serves as a comprehensive guide for drafting a blog post that aims to engage readers and provide in-depth insights into the latest trends in artificial intelligence.